## Part A — Workspace and input verification

In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)
print("UPLOAD the two PDFs into the raw folder now (Files panel > week3_project > data > raw).")

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed
UPLOAD the two PDFs into the raw folder now (Files panel > week3_project > data > raw).


In [2]:
# Cell 3 — Verify uploaded inputs
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)
assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 2
IRBM-e-Invoice-Guideline.pdf bytes: 944059
irbm-e-invoice-specific-guideline.pdf bytes: 2716119
Upload verified. Select your relevant input(s) for Step A3.


In [3]:
# Cell A3 — Inspect both source PDFs
!pip -q install pypdf
from pypdf import PdfReader
import re

SOURCES = {
    "S01": "IRBM-e-Invoice-Guideline.pdf",
    "S02": "irbm-e-invoice-specific-guideline.pdf",
}

for sid, name in SOURCES.items():
    path = RAW_DIR / name
    reader = PdfReader(str(path))
    pages = [(p.extract_text() or "") for p in reader.pages]
    lens = [len(t) for t in pages]
    empty = [i + 1 for i, n in enumerate(lens) if n < 50]

    print("=" * 70)
    print(sid, name)
    print("Pages:", len(pages))
    print("Total characters:", sum(lens))
    print("Pages with almost no text (<50 chars):", empty[:20], "| count:", len(empty))

    head = " ".join(pages[:3])
    print("Version clues:", re.findall(r"(?i)version\s*[\d.]+|[\d]{1,2}\s+\w+\s+20\d{2}", head)[:6])

    mid = len(pages) // 2
    print("\n--- Page 1 sample ---")
    print(repr(pages[0][:400]))
    print(f"\n--- Page {mid + 1} sample ---")
    print(repr(pages[mid][:400]))

    lines = [l.strip() for t in pages for l in t.splitlines() if l.strip()]
    repeated = [l for l in set(lines) if lines.count(l) >= max(5, len(pages) // 4)]
    print("\nPossible repeated header/footer lines:", repeated[:5])

S01 IRBM-e-Invoice-Guideline.pdf
Pages: 62
Total characters: 75704
Pages with almost no text (<50 chars): [] | count: 0
Version clues: ['VERSION 4.8', '30 AUGUST 2026', 'VERSION 4.8', 'VERSION 4.8', 'Version 4.8', 'Version 4.7']

--- Page 1 sample ---
'E-INVOICE GUIDELINE (VERSION 4.8) \n1 \n \n \n \n \n \n \nE-INVOICE GUIDELINE \nINLAND REVENUE BOARD OF MALAYSIA \nThis guideline is issued under section 134A of the Income Tax Act 1967 \n \n(DATE OF PUBLICATION: 30 AUGUST 2026) \n \n \nTABLE OF CONTENT \n \nTABLE OF CONTENT ..................................................................................................... 1 \nSUMMARY OF CHANGES .................'

--- Page 32 sample ---
'E-INVOICE GUIDELINE (VERSION 4.8) \n32 \n \n2.4.3  Step 2 – e-Invoice Validation \nOnce validated by the MyInvois System (which is done in near real-time), \nthe Supplier or technology provider (if Supplier utili ses a technology \nprovider) will receive an API response which contains the following:  

## Part B — Common helpers, chunking comparison, prepared export, lineage

In [4]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully.")
print("Available functions: select_source(), source_hash(), save_manifest()")

Cell 4 completed successfully.
Available functions: select_source(), source_hash(), save_manifest()


In [5]:
# CELL 4A — COMMON LINEAGE REGISTRY (run ONCE, before Cell 5A2)
from pathlib import Path
lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    key = (str(source), str(output))
    lineage_entries[key] = {"source": source, "output": output, "processing": processing}
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


In [6]:
# Cell B3 — Compare chunking approaches on S01 and S02 (evidence for the decisions file)
SOURCES = {
    "S01": "IRBM-e-Invoice-Guideline.pdf",
    "S02": "irbm-e-invoice-specific-guideline.pdf",
}
SIZE = 500
HEADER = re.compile(r"E-INVOICE (SPECIFIC )?GUIDELINE \(VERSION [\d.]+\)", re.I)

def fixed_chunks(s, size=SIZE):
    return [s[i:i + size] for i in range(0, len(s), size)]

def paragraph_chunks(s, size=SIZE):
    parts, current = [], ""
    for seg in re.split(r"\n\s*\n", s):
        seg = seg.strip()
        if not seg:
            continue
        if len(seg) > size:
            if current:
                parts.append(current); current = ""
            parts.extend(fixed_chunks(seg, size))
        elif len(current) + len(seg) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + seg
        else:
            if current:
                parts.append(current)
            current = seg
    if current:
        parts.append(current)
    return parts

def clean_page(t):
    t = HEADER.sub("", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"^\s*\d+\s*$", "", t, flags=re.M)
    return t.strip()

for sid, name in SOURCES.items():
    reader = PdfReader(str(RAW_DIR / name))
    pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]

    whole = re.sub(r"[ \t]+", " ", "\n".join(t for _, t in pages)).strip()
    fixed = fixed_chunks(whole)

    paged = []
    for pno, t in pages:
        for c in paragraph_chunks(clean_page(t)):
            paged.append({"page": pno, "text": c})

    print("=" * 70)
    print(sid, name)
    print("A fixed-size chunks:", len(fixed), "| avg length:", round(sum(map(len, fixed)) / len(fixed)))
    print("B page-aware chunks:", len(paged),
          "| avg length:", round(sum(len(c["text"]) for c in paged) / len(paged)),
          "| pages covered:", len({c["page"] for c in paged}), "of", len(pages))
    k = 3
    print("\nA boundary (end of chunk 3 | start of chunk 4):")
    print(repr(fixed[k - 1][-70:]), "|", repr(fixed[k][:70]))
    mid = paged[len(paged) // 2]
    print(f"\nB sample chunk (page {mid['page']}):")
    print(repr(mid["text"][:250]))

S01 IRBM-e-Invoice-Guideline.pdf
A fixed-size chunks: 151 | avg length: 499
B page-aware chunks: 194 | avg length: 373 | pages covered: 62 of 62

A boundary (end of chunk 3 | start of chunk 4):
'................ 10 \n1.4 Scenarios and Types of e-Invoices ...........' | '..................................................... 10 \n1.5 e-Invoic'

B sample chunk (page 30):
'The summary of the e -Invoice model flow via API is diagrammatically depicted \nin Figure 2.11.\n\nFigure 2.11 – e-Invoice model workflow via API'
S02 irbm-e-invoice-specific-guideline.pdf
A fixed-size chunks: 307 | avg length: 499
B page-aware chunks: 382 | avg length: 382 | pages covered: 132 of 132

A boundary (end of chunk 3 | start of chunk 4):
'............................................................. 63 \n9. T' | 'RANSACTIONS WHICH INVOLVE PAYMENTS (WHETHER IN MONETARY \nFORM OR OTHER'

B sample chunk (page 64):
'iii. Interest payment made by foreign payor to Malaysian taxpayers; \niv. Interest payment to a re

In [7]:
# Cell 5A2 (adapted) — Page-aware export + validation + lineage for S01 and S02
import json, re
from pypdf import PdfReader

SIZE = 500
HEADER = re.compile(r"E-INVOICE (SPECIFIC )?GUIDELINE \(VERSION [\d.]+\)", re.I)

DOCS = {
    "S01": {"file": "IRBM-e-Invoice-Guideline.pdf", "version": "4.8", "published": "30 August 2026"},
    "S02": {"file": "irbm-e-invoice-specific-guideline.pdf", "version": "4.7", "published": "20 April 2026"},
}
APPROACH = "page-aware paragraph chunks, 500 chars; repeated header and page-number line removed"

def fixed_chunks(s, size=SIZE):
    return [s[i:i + size] for i in range(0, len(s), size)]

def paragraph_chunks(s, size=SIZE):
    parts, current = [], ""
    for seg in re.split(r"\n\s*\n", s):
        seg = seg.strip()
        if not seg:
            continue
        if len(seg) > size:
            if current:
                parts.append(current); current = ""
            parts.extend(fixed_chunks(seg, size))
        elif len(current) + len(seg) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + seg
        else:
            if current:
                parts.append(current)
            current = seg
    if current:
        parts.append(current)
    return parts

def clean_page(t):
    t = HEADER.sub("", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"^\s*\d+\s*$", "", t, flags=re.M)
    return t.strip()

for sid, meta in DOCS.items():
    f = select_source(relative_path=meta["file"])
    reader = PdfReader(str(f))
    sha = source_hash(f)

    records = []
    for pno, page in enumerate(reader.pages, 1):
        for chunk in paragraph_chunks(clean_page(page.extract_text() or "")):
            records.append({
                "record_id": f"{sid}-{len(records) + 1:04d}",
                "text": chunk,
                "source": str(f.relative_to(RAW_DIR)),
                "source_sha256": sha,
                "document_version": meta["version"],
                "published": meta["published"],
                "page": pno,
                "processing": APPROACH,
            })

    prepared_path = PROCESSED_DIR / f"prepared_text_{sid}.jsonl"
    with prepared_path.open("w", encoding="utf-8") as out:
        for r in records:
            out.write(json.dumps(r, ensure_ascii=False) + "\n")

    assert prepared_path.exists() and prepared_path.stat().st_size > 0, "Output missing or empty"
    assert records and all(r["text"].strip() for r in records), "Empty chunk found"
    assert all(r["page"] is not None for r in records), "Missing page number"
    with prepared_path.open(encoding="utf-8") as fh:
        back = [json.loads(l) for l in fh if l.strip()]
    assert len(back) == len(records), "Exported record count mismatch"

    print("=" * 60)
    print(sid, "| version", meta["version"], "|", f.name)
    print("Chunks exported:", len(records), "| pages covered:",
          len({r["page"] for r in records}), "of", len(reader.pages))
    print("File:", prepared_path, "| bytes:", prepared_path.stat().st_size)
    print("Validation: PASS")
    print("Sample:", records[len(records) // 2])

    register_lineage(source=f, output=prepared_path, processing=APPROACH)

print("\nRegistered entries:", len(lineage_entries))

S01 | version 4.8 | IRBM-e-Invoice-Guideline.pdf
Chunks exported: 194 | pages covered: 62 of 62
File: /content/week3_project/data/processed/prepared_text_S01.jsonl | bytes: 139417
Validation: PASS
Sample: {'record_id': 'S01-0098', 'text': 'The summary of the e -Invoice model flow via API is diagrammatically depicted \nin Figure 2.11.\n\nFigure 2.11 – e-Invoice model workflow via API', 'source': 'IRBM-e-Invoice-Guideline.pdf', 'source_sha256': 'af55b6a71e0a81af3813e7495761b6dbd156feb636b451cb66288ad08f8153bc', 'document_version': '4.8', 'published': '30 August 2026', 'page': 30, 'processing': 'page-aware paragraph chunks, 500 chars; repeated header and page-number line removed'}
Registered: IRBM-e-Invoice-Guideline.pdf -> prepared_text_S01.jsonl
S02 | version 4.7 | irbm-e-invoice-specific-guideline.pdf
Chunks exported: 382 | pages covered: 132 of 132
File: /content/week3_project/data/processed/prepared_text_S02.jsonl | bytes: 282531
Validation: PASS
Sample: {'record_id': 'S02-0192', 'te

In [8]:
# CELL 6A — Generate common lineage manifest for S01 and S02
SOURCES_META = {
    "S01": {"file": "IRBM-e-Invoice-Guideline.pdf", "version": "4.8", "published": "30 August 2026"},
    "S02": {"file": "irbm-e-invoice-specific-guideline.pdf", "version": "4.7", "published": "20 April 2026"},
}
PROCESSING = "page-aware paragraph chunks, 500 chars; repeated header and page-number line removed"

for sid, meta in SOURCES_META.items():
    source = (RAW_DIR / meta["file"]).resolve()
    output = (PROCESSED_DIR / f"prepared_text_{sid}.jsonl").resolve()
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing prepared output: {output.name}. Run Cell 5A2 first."
    assert output.stat().st_size > 0, f"Empty output: {output.name}"
    lineage_entries[(str(source), str(output))] = {"source": source, "output": output, "processing": PROCESSING}

assert len(lineage_entries) == len(SOURCES_META), (
    f"Expected {len(SOURCES_META)} lineage entries, found {len(lineage_entries)}. "
    "Run lineage_entries.clear() and rerun Cell 5A2 and this cell if old entries remain."
)

lineage_records = []
for sid, meta in SOURCES_META.items():
    source = (RAW_DIR / meta["file"]).resolve()
    output = (PROCESSED_DIR / f"prepared_text_{sid}.jsonl").resolve()
    entry = lineage_entries[(str(source), str(output))]
    lineage_records.append({
        "source_id": sid,
        "source_filename": str(entry["source"].relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(entry["source"]),
        "prepared_output": str(entry["output"].relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(entry["output"]),
        "processing_note": entry["processing"],
        "document_version": meta["version"],
        "published": meta["published"],
        "source_version": f"{meta['version']} (published {meta['published']}; read from PDF cover, not independently verified)",
        "access_rights": "Public on hasil.gov.my; redistribution terms not verified",
        "status": "Prepared; quality review pending",
    })

manifest = save_manifest(lineage_records)
print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print("Saved to:", BASE_DIR / "data" / "week3_manifest.json")

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 2
Saved to: /content/week3_project/data/week3_manifest.json


In [9]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
import json

manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

saved_records = saved_manifest.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."

for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared output checksum mismatch: {output}"

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 2
Manifest: /content/week3_project/data/week3_manifest.json


## Part C — V1 data pipeline

In [10]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"
PIPE_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_RUNS = {}

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(f"Source folder not found: {source}")
    else:
        if not source.is_file():
            raise FileNotFoundError(f"Source file not found: {source}")
    return source

def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)

    # INGEST
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
        )
    else:
        inputs = [source]
    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")

    # INSPECT
    inspection = inspect_fn(inputs, configuration)

    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]

    # VALIDATE
    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")

    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs],
    }

    # RECORD LINEAGE
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": str(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": "Not verified — check before sharing",
    }

    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", manifest_path)
    return record

def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [11]:
# Cell 8A (adapted) — PDF pipeline for S01 and S02, page-aware chunks
# Requires Cell 7
!pip -q install pypdf
import re
import json
from pypdf import PdfReader

HEADER = re.compile(r"E-INVOICE (SPECIFIC )?GUIDELINE \(VERSION [\d.]+\)", re.I)

DOCS = {
    "S01": {"file": "IRBM-e-Invoice-Guideline.pdf", "version": "4.8", "published": "30 August 2026"},
    "S02": {"file": "irbm-e-invoice-specific-guideline.pdf", "version": "4.7", "published": "20 April 2026"},
}

def fixed_chunks(s, size):
    return [s[i:i + size] for i in range(0, len(s), size)]

def paragraph_chunks(s, size):
    parts, current = [], ""
    for seg in re.split(r"\n\s*\n", s):
        seg = seg.strip()
        if not seg:
            continue
        if len(seg) > size:
            if current:
                parts.append(current); current = ""
            parts.extend(fixed_chunks(seg, size))
        elif len(current) + len(seg) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + seg
        else:
            if current:
                parts.append(current)
            current = seg
    if current:
        parts.append(current)
    return parts

def clean_page(t):
    t = HEADER.sub("", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"^\s*\d+\s*$", "", t, flags=re.M)
    return t.strip()

def inspect_documents(inputs, cfg):
    src = inputs[0]
    reader = PdfReader(str(src))
    pages = [p.extract_text() or "" for p in reader.pages]
    if not any(t.strip() for t in pages):
        raise ValueError("No extractable text found. Scanned PDFs may require OCR.")
    return {
        "source": src.name,
        "source_id": cfg["source_id"],
        "document_version": cfg["document_version"],
        "pages": len(pages),
        "characters": sum(len(t) for t in pages),
    }

def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    reader = PdfReader(str(src))
    size = int(cfg["chunk_size"])
    sha = sha256_file(src)

    records = []
    for pno, page in enumerate(reader.pages, 1):
        for chunk in paragraph_chunks(clean_page(page.extract_text() or ""), size):
            records.append({
                "record_id": f"{cfg['source_id']}-{len(records) + 1:04d}",
                "text": chunk,
                "source": src.name,
                "source_sha256": sha,
                "document_version": cfg["document_version"],
                "published": cfg["published"],
                "page": pno,
            })
    if not records:
        raise ValueError("No text chunks produced")

    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for r in records:
            handle.write(json.dumps(r, ensure_ascii=False) + "\n")

    mapping = [{"source": str(src), "output": str(target)}]
    method = f"page-aware paragraph chunks of up to {size} characters; header and page-number line removed"
    return [target], mapping, method

for sid, meta in DOCS.items():
    config = {
        "source_id": sid,
        "chunk_size": 500,
        "method": "page-aware paragraph",
        "document_version": meta["version"],
        "published": meta["published"],
    }
    register_and_run(
        f"documents_{sid}",
        resolve_source(meta["file"]),
        inspect_documents,
        prepare_documents,
        config,
    )

print("Cell 8A completed successfully for", list(DOCS))

SUCCESS: documents_S01 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_S01/pipeline_manifest.json
SUCCESS: documents_S02 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_S02/pipeline_manifest.json
Cell 8A completed successfully for ['S01', 'S02']


In [12]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
assert PIPELINE_RUNS, "No pipeline runs registered. Run Cell 8A first."

print("--- Repeatability Verification ---")
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    second = entry["runner"]()
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    assert first["source_output_mapping"] == second["source_output_mapping"], f"Mapping changed for {kind}"
    print(f"PASS: {kind} — source fingerprints, prepared outputs and mappings match")

print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents_S01 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_S01/pipeline_manifest.json
PASS: documents_S01 — source fingerprints, prepared outputs and mappings match
SUCCESS: documents_S02 | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_S02/pipeline_manifest.json
PASS: documents_S02 — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.
